# irsBookToIRS — Fill Form8825

A **single function** — `BookToIRS('Form8825')` — that turns the LLC's
financial books into `Form1065_FILL.pdf`, using only the public
service APIs:

* **Data side (book values)** — `stmtProfile`, `stmtGL_Tax`,
  `stmtBS_Tax`, `stmtIS_Tax`.  Each exposes `loadFillDict(formNm)`
  returning `{normalized_fid → fval}`, where the fid is in canonical
  zero-padded form `F###` and `fval` is one of:
    * a resolved book value (str or number) — **filled**;
    * `None` / `""`                          — **blank** (recognised
      but not yet sourced; *not* an error);
    * the literal string `'Complex'`         — bookNS UAS is tagged
      `Cplx*`, meaning the field needs multi-source composition done
      in a follow-up task.
* **PDF side (form layout)** — `irs.Form1065`.  All knowledge of where
  the IRS template / namespace JSON / FILL.pdf live is encapsulated
  here.  The notebook never reaches into IRS internals.  The relevant
  helpers are:
    * `f1065.loadFieldsDF()`     — DataFrame of every fillable PDF
      field with normalized `fid`;
    * `f1065.saveFILL_FromDF(df)` — write the FILL.pdf from a
      DataFrame keyed by `fid`.

The notebook merges the two sides with a single **pandas** `df.merge`
on the normalized `fid` — no hand-rolled dict merging, no
fid-translation logic in the notebook itself.

## Goals

1. **Maximize the values filled** into `Form1065_FILL.pdf`.
2. Provide a **reconciliation aid** — a side-by-side table that lets a
   human compare `Form1065_namespace.pdf` vs `Form1065_FILL.pdf` and
   spot wrong / missing values, then fix them in the relevant
   `stmtOBJ` code or `bookNS_*.json`.
3. **Distinguish blank from Complex** — *most* IRS fields should
   legitimately stay empty.  `'Complex'` is reserved for fields that
   need real composition work (e.g. multi-source aggregation).

The function returns:
```python
{
    'fillDict':    { fid: row_dict, ... },   # status == 'filled'
    'complexDict': { fid: row_dict, ... },   # status == 'complex'
    'df':          pandas.DataFrame          # full per-field reconciliation
}
```


## 1. Init — import services and build the LLC.

In [1]:
# Make ledger/ and irs/ importable from the Notebooks/ root.
import os, sys
from pathlib import Path

_NB_DIR = Path.cwd()
if (_NB_DIR / 'ledger').is_dir() and str(_NB_DIR) not in sys.path:
    sys.path.insert(0, str(_NB_DIR))

import pandas as pd

from ledger.LLC          import LLC
from ledger.stmtProfile  import stmtProfile
from ledger.stmtGL       import stmtGL_Tax
from ledger.stmtBS       import stmtBS_Tax
from ledger.stmtIS       import stmtIS_Tax
from irs.Form1065        import Form1065

llc = LLC('WBGroupLLC')
print(f"LLC : {llc.entity.get('entity_name')}  EIN={llc.entity.get('ein')}  Year={llc.yr}")


LLC : W&B Group, LLC  EIN=39-3842347  Year=2025


## 2. `BookToIRS(formNm)` — the only function.

The pipeline in 6 short steps:

1. Instantiate the IRS service (`Form1065`) and the four book services
   (`stmtProfile`, `stmtGL_Tax`, `stmtBS_Tax`, `stmtIS_Tax`).
2. Build one **book DataFrame** by stacking each stmt's `loadFillDict`
   output (each row: `fid, fval, source`).
3. Resolve duplicates with **priority Profile > BS > IS > GL** —
   pandas `sort + drop_duplicates(fid, keep='first')`.
4. Build the **fields DataFrame** from `f1065.loadFieldsDF()` (one row
   per fillable AcroForm field, normalized fid).
5. **Merge** the two DataFrames on `fid` (left join) and label every
   row `filled` / `complex` / `blank`.
6. Hand the merged DataFrame to `f1065.saveFILL_FromDF(df)` to write
   the FILL.pdf.  Done.

In [2]:
from irs.BookToIRS import BookToIRS
b = BookToIRS(llc, formNm='Form8825')

bkList = b.listSources()
bkDict = {oNm:{"ns":b.loadBookNS(oNm),"cplx":b.loadCustomMapDict(oNm)} for oNm in bkList}
nsTxDict = b.loadNamespace()


In [3]:

class irsFormCustom(Form1065):
    pass
    def _has_value(self, v):
        if v is None: return False
        if isinstance(v, str): return v != ''
        try:
            return not pd.isna(v)
        except Exception:
            return True

    def _status(self, v):
        if isinstance(v, str) and v == 'Complex':
            return 'complex'
        if isinstance(v, str) and v == 'CHECK':
            return 'check'
        if v is None:
            return 'blank'
        try:
            if pd.isna(v):
                return 'blank'
        except Exception:
            pass
        if isinstance(v, str) and v == '':
            return 'blank'
        return 'filled'

    def _value(self, row):
        s = row['status']
        if s == 'filled':
            v = row['fval']
            return v if isinstance(v, str) else f"{v}"
        if s == 'check':
            return 'CHECK'         # saveFILL_FromDF stamps X (text) or toggles /Btn
        if s == 'complex':
            return 'Complex'
        return None                # blank â do not write


    def BookToIRS(self, formNm: str) -> dict:
        '''
        Build `<formNm>_FILL.pdf` from book data via the public service APIs.
    
        Returns
        -------
        dict   { 'fillDict':    {fid: row_dict, ...},   # status = 'filled'
                 'checkDict':   {fid: row_dict, ...},   # status = 'check'  (X / checkbox)
                 'complexDict': {fid: row_dict, ...},   # status = 'complex'
                 'df':          DataFrame              }   # full reconciliation table
        '''
        # ââ 1. Services ââ
        f1065 = Form1065(llc=llc)
        book_sources = [
            ('Profile', stmtProfile (llc).loadFillDict(formNm)),
            ('BS',      stmtBS_Tax  (llc).loadFillDict(formNm)),
            ('IS',      stmtIS_Tax  (llc).loadFillDict(formNm)),
            ('GL',      stmtGL_Tax  (llc).loadFillDict(formNm)),
        ]
    
        # ââ 2. Stack book values into one DF (rows: fid / fval / source) ââ
        book_rows = []
        for src, d in book_sources:
            for fid, fval in d.items():
                book_rows.append({'fid': fid, 'fval': fval, 'source': src})
        df_book = pd.DataFrame(book_rows, columns=['fid', 'fval', 'source'])
    
        # ââ 3. Priority resolution (Profile > BS > IS > GL).  'has a real value'
        #       wins, so a Profile None loses to a BS non-empty for the same fid.
        df_book['has_val'] = df_book['fval'].apply(self._has_value)
        _PRIO = {'Profile': 0, 'BS': 1, 'IS': 2, 'GL': 3}
        df_book['_prio'] = df_book['source'].map(_PRIO).fillna(99).astype(int)
        df_book = (df_book
                   .sort_values(['fid', 'has_val', '_prio'],
                                ascending=[True, False, True])
                   .drop_duplicates('fid', keep='first')
                   .drop(columns=['has_val', '_prio'])
                   .reset_index(drop=True))
    
        # ââ 4. Fields DF from the IRS service ââ
        df_fields = f1065.loadFieldsDF()
        if df_fields.empty:
            raise RuntimeError(f"{formNm}: loadFieldsDF returned no fields")
    
        # ââ 5. Merge & classify ââ
        df = df_fields.merge(df_book, on='fid', how='left')
    
        df['status'] = df['fval'].apply(self._status)
    
        df['value'] = df.apply(self._value, axis=1)
    
        # ââ 6. Write FILL.pdf via the IRS service ââ
        out_path = f1065.saveFILL_FromDF(df)
    
        # ââ Reporting ââ
        n_total   = len(df)
        n_filled  = int((df['status'] == 'filled' ).sum())
        n_check   = int((df['status'] == 'check'  ).sum())
        n_complex = int((df['status'] == 'complex').sum())
        n_blank   = int((df['status'] == 'blank'  ).sum())
        print(f"✅  {formNm}_FILL.pdf  ->  {out_path}")
        print(f"   pdf_fields={n_total}   filled={n_filled}   "
              f"check={n_check}   complex={n_complex}   blank={n_blank}")
    
        fillDict    = (df[df['status'] == 'filled' ]
                       .set_index('fid').to_dict(orient='index'))
        checkDict   = (df[df['status'] == 'check'  ]
                       .set_index('fid').to_dict(orient='index'))
        complexDict = (df[df['status'] == 'complex']
                       .set_index('fid').to_dict(orient='index'))
        return {'fillDict': fillDict, 'checkDict': checkDict,
                'complexDict': complexDict, 'df': df}


# Get property info for lines A-D

In [4]:
from ledger.llcAssets import llcAssets

aObj = llcAssets(llc)
aList = aObj.load()
aDF = pd.DataFrame(pList)
aDF['pKey'] = aDF.propOwners.apply(lambda v : '.'.join(list(v.keys())))
pDF = aDF[aDF.pKey == 'LLC'].copy().set_index('propNm', drop=True) #.groupby(['propOwners','propNm','propAddr']).propNm.count()
pDF

NameError: name 'pList' is not defined

bookNS_IS = { "Form8825": [
    [
      "F023",
      "IS.rent_income"
    ],
    [
      "F071",
      "IS.depreciation"
    ]
  ]
  }

In [9]:
#from ledger.stmtGL import stmtGL
from ledger.stmtGeneralLedger import stmtGeneralLedger as stmtGL
gl = stmtGL(llc)

'''
Build Income Statement raw table for downstream consumption
- balance is divided by property (propNm)
- FIXME : PropNm is not in GL so it can not be used,  enhance by: ledgerObject.toGL(cols=None)

'''
glObj = stmtGL(llc)
glList = glObj.load()
glDF = pd.DataFrame(glList)
#glDF.acctType = glDF.acct.apply(erObj.llc.coa._Type)

isDF = glDF.groupby(['acctType', 'acct','propNm','aType']).amt.sum().unstack()\
       .fillna(0)

isDF['Balance'] = isDF.Debit - isDF.Credit
isDF.drop(columns=['Credit','Debit'], inplace=True)
sumDF = isDF.unstack().reset_index()
sumDF.columns = [c[0] if c[1] == '' else c[1] for c in sumDF.columns]
sumDF = sumDF.set_index('acct', drop=True).fillna('')


,acctType,Cash_LLC,H_805HighMesa,RV_RV1
acct,,,,
Acct.AR.Loan,Asset,0.0,,
Acct.Cash.Bank,Asset,0.0,-217742.87,-857.01
Acct.Fixed.Depreciation.Accum,Asset,,-5246.06,
Acct.Fixed.Tangible.InConstruction,Asset,,462.15,810.13
Acct.Fixed.Tangible.InService,Asset,,437950.81,
Acct.Fixed.Tangible.LongTerm,Bad_Acct.Fixed.Tangible.LongTerm,,,712.14
Acct.Equity.Earnings.PnL,Equity,0.0,,
Acct.Equity.Owner.Capital.Funds,Equity,,-218108.51,-987.13
Acct.Exp.Depreciation,Expense,,5246.06,


In [ ]:
from ledger.stmtIS import stmtIS_Tax, stmtIS
isObj = stmtIS(llc)
isList = isObj.load()
isTxObj = stmtIS_Tax(llc)
isTxList = isObj.load()
pd.DataFrame(isList)

In [6]:
from ledger.stmtGeneralLedger import stmtGeneralLedger as stmtGL

glObj = stmtGL(llc)
glList = glObj.load()
df = pd.DataFrame(glList)
df

,Status,dt,acctType,acct,acctMinor,propNm,aType,amt,desc,acctSub,refDB,tID,_lineNo,_rowNm
0,⚠ Dup,2025.08.20,Asset,Acct.AR.Loan,Loan,Cash_LLC,Debit,0.0,Null Transaction - template,,llcReceivables,2025.08.20_0.00,1,2025.08.20_0.00
1,⚠ Dup,2025.08.20,Asset,Acct.AR.Loan,Loan,Cash_LLC,Credit,0.0,Null Transaction - template,,llcReceivables,2025.08.20_-0.00,2,2025.08.20_-0.00
2,⚠ Dup,2025.08.20,Asset,Acct.Cash.Bank,Bank,Cash_LLC,Debit,0.0,YR.2025.Begining Balance: Cash,Beg Bal,llcBank,2025.08.20_0.00,3,2025.08.20_0.00#2
3,⚠ Dup,2025.08.20,Asset,Acct.Cash.Bank,Bank,H_805HighMesa,Debit,50.0,Open Bank Acct Investment,Open Bank Acct,llcAssets,2025.08.20_50.00,4,2025.08.20_50.00
4,,2025.08.20,Asset,Acct.Cash.Bank,Bank,H_805HighMesa,Debit,219000.0,Owner investment,Closing-805 High Mesa,llcBank,2025.08.20_219000.00,5,2025.08.20_219000.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123,,2025.12.26,Expense,Acct.Exp.Util,Util,H_805HighMesa,Debit,135.8,Pay Monthly Util,Ins_Home,llcBank,2025.12.26_135.80,124,2025.12.26_135.80
124,⚠ Dup,2025.12.29,Asset,Acct.Cash.Bank,Bank,RV_RV1,Debit,177.0,Rental RV1 - Purchase,NewRV,llcAssets,2025.12.29_177.00,125,2025.12.29_177.00
125,⚠ Dup,2025.12.29,Asset,Acct.Cash.Bank,Bank,H_805HighMesa,Debit,177.0,Owner Investment,,llcBank,2025.12.29_177.00,126,2025.12.29_177.00#2
126,⚠ Dup,2025.12.29,Equity,Acct.Equity.Owner.Capital.Funds,Owner.Capital.Funds,RV_RV1,Credit,177.0,Rental RV1 - Purchase,NewRV,llcAssets,2025.12.29_-177.00,127,2025.12.29_-177.00


In [9]:
from ledger.stmtGL import stmtGL
glObj = stmtGL(llc)
glList = glObj.load()
df = pd.DataFrame(glList)
df.propNm.unique() #.groupby(['acctType', 'acct','propNm','aType']).sum().unstack()

AttributeError: 'DataFrame' object has no attribute 'propNm'

In [31]:
df[df.acct == 'Acct.Rev.Rent']

,Ledger,_unknown,aType,acct,acctSub,amt,desc,dt,propAddr,propID,propNm,propOwners,refDB,refDoc,tDB,tID,acctType


## 3. Run `BookToIRS('Form1065')`.

In [6]:
from irs.irsForm import irsForm
from irs.Form8825 import Form8825
from irs.BookToIRS import BookToIRS
f = Form8825(llc)

nspace   = f._buildNSpace()
#f.saveNSpace(nspace)
fillDict = f._buildFillDict(nspace)   # publish + value resolution

irsForm.Form8825 buildFillDict Entry


In [9]:
pd.DataFrame(fillDict).transpose()

,fID,pdfField,shortName,logicalKey,label,fType,page,location,checkedValue,publish,source,path,note,value
f1,f1,topmostSubform[0].Page1[0].f1_1[0],f1_1,,,text,1,Form8825.Pg1.Unknown,/1,False,None,None,,
f2,f2,topmostSubform[0].Page1[0].f1_2[0],f1_2,,,text,1,Form8825.Pg1.Unknown,/1,False,None,None,,
f3,f3,topmostSubform[0].Page1[0].Table_Line1[0].RowA...,f1_3,,,text,1,Form8825.Pg1.Unknown,/1,False,None,None,,
f4,f4,topmostSubform[0].Page1[0].Table_Line1[0].RowA...,f1_4,,,text,1,Form8825.Pg1.Unknown,/1,False,None,None,,
f5,f5,topmostSubform[0].Page1[0].Table_Line1[0].RowA...,f1_5,,,text,1,Form8825.Pg1.Unknown,/1,False,None,None,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
f217,f217,topmostSubform[0].Page2[0].Table_Lines2-17[0]....,f2_104,,,text,2,Form8825.Pg2.Unknown,/1,False,None,None,,
f218,f218,topmostSubform[0].Page2[0].Table_Lines2-17[0]....,f2_105,,,text,2,Form8825.Pg2.Unknown,/1,False,None,None,,
f219,f219,topmostSubform[0].Page2[0].Table_Lines2-17[0]....,f2_106,,,text,2,Form8825.Pg2.Unknown,/1,False,None,None,,
f220,f220,topmostSubform[0].Page2[0].Table_Lines2-17[0]....,f2_107,,,text,2,Form8825.Pg2.Unknown,/1,False,None,None,,


In [ ]:
f.saveFILL(fillDict)

In [ ]:
result      = BookToIRS('Form8825')
fillDict    = result['fillDict']
checkDict   = result['checkDict']
complexDict = result['complexDict']
df          = result['df']

print(f"\nfillDict ({len(fillDict)}) — first 5:")
for fid in list(fillDict)[:5]:
    r = fillDict[fid]
    print(f"  {fid}  src={r['source']:<7} value={r['value']!r:<30}  "
          f"shortName={r['shortName']!r}  page={r['page']}")

print(f"\ncheckDict ({len(checkDict)}) — first 5:")
for fid in list(checkDict)[:5]:
    r = checkDict[fid]
    print(f"  {fid}  src={r['source']!r:<10} shortName={r['shortName']!r}  "
          f"fType={r['fType']:<10} page={r['page']}")

print(f"\ncomplexDict ({len(complexDict)}) — first 5:")
for fid in list(complexDict)[:5]:
    r = complexDict[fid]
    print(f"  {fid}  src={r['source']!r:<10} shortName={r['shortName']!r}  page={r['page']}")

AttributeError: 'irsForm' object has no attribute 'BookToIRS'

✅  Form1065_FILL.pdf  ->  /tmp/fake_home/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/pages/AccountingData/2025/YE_Tax_Records/Forms_IRS/Form1065_FILL.pdf
   pdf_fields=440   filled=55   check=36   complex=0   blank=349

fillDict (55) — first 5:
  F001  src=Profile value='Jan. 01'                       shortName='f1_01'  page=1
  F002  src=Profile value='Dec. 31'                       shortName='f1_02'  page=1
  F004  src=Profile value='W&B Group, LLC'                shortName='f1_04'  page=1
  F005  src=Profile value='177 Kingsway Dr'               shortName='f1_05'  page=1
  F007  src=Profile value='Wimberley'                     shortName='f1_07'  page=1

checkDict (36) — first 5:
  F017  src='Profile'  shortName='f1_17'  fType=text       page=1
  F021  src='Profile'  shortName='f1_21'  fType=text       page=1
  F080  src='Profile'  shortName='f2_02'  fType=text       page=2
  F086  src='Profile'  shortName='f2_08'  fType=text       page=2
  F087  src='Profile'  sho

## 4. Reconciliation aid

The merged DataFrame `df` is the single point of truth.  Every PDF
field appears exactly once with its normalized fid, the source stmt
that supplied a value (if any), the resolved value, and a status.

Use this table side-by-side with `Form1065_namespace.pdf` (which shows
every PDF field labelled with its native `f###` fid) and the freshly
written `Form1065_FILL.pdf` to spot fields that look wrong.  When you
find one, fix the underlying stmtOBJ code or `bookNS_*.json` mapping
and re-run `BookToIRS('Form1065')`.

In [4]:
# Status mix per source
print("Status counts per book source:")
print(df.groupby(['source', 'status']).size().unstack(fill_value=0))

# Sample of filled rows
print("\nFilled — first 12 rows:")
print(df[df['status'] == 'filled']
      [['fid', 'pdf_fid', 'shortName', 'page', 'source', 'value']]
      .head(12)
      .to_string(index=False))

# Sample of complex rows (will be empty until bookNS Cplx-tags are added)
print("\nComplex — first 8 rows:")
cmplx = df[df['status'] == 'complex'][['fid','pdf_fid','shortName','page','source']]
print(cmplx.head(8).to_string(index=False) if len(cmplx) else "  (none)")


Status counts per book source:
status   blank  check  filled
source                       
BS           0      0      20
IS           0      0      22
Profile     59     36      13

Filled — first 12 rows:
 fid pdf_fid shortName  page  source           value
F001      f1     f1_01     1 Profile         Jan. 01
F002      f2     f1_02     1 Profile         Dec. 31
F004      f4     f1_04     1 Profile  W&B Group, LLC
F005      f5     f1_05     1 Profile 177 Kingsway Dr
F007      f7     f1_07     1 Profile       Wimberley
F008      f8     f1_08     1 Profile              Tx
F009      f9     f1_09     1 Profile             USA
F010     f10     f1_10     1 Profile           78676
F011     f11     f1_11     1 Profile          531110
F012     f12     f1_12     1 Profile          531110
F013     f13     f1_13     1 Profile          531110
F014     f14     f1_14     1 Profile      39-3842347

Complex — first 8 rows:
  (none)


In [14]:
print(llc.F1065['chk'])

[17, 21, 80, 86, 87, 90, 91, 112, 138, 141, 143, 145, 148, 150, 153, 157, 161, 164, 166, 169, 173, 176, 178, 182, 184, 186, 188, 191, 193, 195, 199, 203, 205, 207, 209, 212]


In [18]:
llc.F1065

{'address': '177 Kingsway Dr.',
 'A_bus_act': '531110',
 'B_product': '531110',
 'C_busCode': '531110',
 'C_city': 'Wimberley',
 'C_state': 'Tx',
 'C_ctry': 'USA',
 'C_zip': '78676',
 'preparer_name': '',
 'preparer_ptin': '',
 'preparer_date': '',
 'preparer_ein': '',
 'preparer_firm': '',
 'preparer_addr': '',
 'preparer_phone': '',
 'principal_activity': 'Property Rental/Management/Investment',
 'B_PRDI_FirstNm': 'Francis X.',
 'B_PRDI_Last': 'Rojas',
 'B_PRDI_Street': '177 Kingsway Dr.',
 'B_PRDI_City': 'Wimberley',
 'B_PRDI_St': 'Tx',
 'B_PRDI_Zip': '78676',
 'B_PRDI_Ph': '512-422-0210',
 'business_code': 'na',
 'date_from': 'Jan. 01',
 'date_to': 'Dec. 31',
 'total_assets': None,
 'number_of_k1s': None,
 'tax_year': None,
 'chk': [17,
  21,
  80,
  86,
  87,
  90,
  91,
  112,
  138,
  141,
  143,
  145,
  148,
  150,
  153,
  157,
  161,
  164,
  166,
  169,
  173,
  176,
  178,
  182,
  184,
  186,
  188,
  191,
  193,
  195,
  199,
  203,
  205,
  207,
  209,
  212]}

In [17]:
df.set_index('fid', drop=True).loc['F021']
df.head(25)

,fid,pdf_fid,pdfField,shortName,fType,pdfFT,page,checkedValue,fval,source,status,value
0,F001,f1,topmostSubform[0].Page1[0].HeaderAddress_ReadO...,f1_01,text,/Tx,1,/1,Jan. 01,Profile,filled,Jan. 01
1,F002,f2,topmostSubform[0].Page1[0].HeaderAddress_ReadO...,f1_02,text,/Tx,1,/1,Dec. 31,Profile,filled,Dec. 31
2,F003,f3,topmostSubform[0].Page1[0].HeaderAddress_ReadO...,f1_03,text,/Tx,1,/1,None,Profile,blank,None
3,F004,f4,topmostSubform[0].Page1[0].HeaderAddress_ReadO...,f1_04,text,/Tx,1,/1,"W&B Group, LLC",Profile,filled,"W&B Group, LLC"
4,F005,f5,topmostSubform[0].Page1[0].HeaderAddress_ReadO...,f1_05,text,/Tx,1,/1,177 Kingsway Dr,Profile,filled,177 Kingsway Dr
5,F006,f6,topmostSubform[0].Page1[0].HeaderAddress_ReadO...,f1_06,text,/Tx,1,/1,None,Profile,blank,None
6,F007,f7,topmostSubform[0].Page1[0].HeaderAddress_ReadO...,f1_07,text,/Tx,1,/1,Wimberley,Profile,filled,Wimberley
7,F008,f8,topmostSubform[0].Page1[0].HeaderAddress_ReadO...,f1_08,text,/Tx,1,/1,Tx,Profile,filled,Tx
8,F009,f9,topmostSubform[0].Page1[0].HeaderAddress_ReadO...,f1_09,text,/Tx,1,/1,USA,Profile,filled,USA
9,F010,f10,topmostSubform[0].Page1[0].HeaderAddress_ReadO...,f1_10,text,/Tx,1,/1,78676,Profile,filled,78676


## 5. `_Cplx_<fid>()` stub methods

For every fid in `complexDict`, emit a stub that documents the field
and what is needed.  No solution code — that is the next task.

Today no `Cplx`-tagged entries exist in `bookNS_*.json`, so
`complexDict` is empty and zero stubs are generated.  As soon as a
human marks a bookNS UAS as `Cplx.<...>` (e.g.
`["F036", "Cplx.line8_total"]`), the stmtOBJ_Tax classes will report
that fid with `fval='Complex'`, the FILL.pdf will be stamped with
`'Complex'` for that field, and a stub will appear here.

In [6]:
def _emit_complex_stubs(complexDict: dict) -> str:
    '''
    Return a Python source-text block defining one stub per complex fid.

    Each stub:
      * is named ``_Cplx_<fid>``  (sanitised: non-word chars -> '_')
      * has a docstring describing the PDF field and what is needed.
      * raises NotImplementedError when called.
    No business logic is generated — filling them in is the next task.
    '''
    import re as _re
    src_lines = []
    for fid, info in complexDict.items():
        safe = _re.sub(r'[^A-Za-z0-9_]', '_', str(fid))
        nm   = f'_Cplx_{safe}'
        src_lines.append(
            f"def {nm}():\n"
            f"    '''\n"
            f"    Complex field stub - {fid!r}.\n"
            f"\n"
            f"    PDF field   : pdf_fid={info.get('pdf_fid')!r}  "
            f"shortName={info.get('shortName')!r}  page={info.get('page')}  "
            f"fType={info.get('fType')!r}  pdfFT={info.get('pdfFT')!r}\n"
            f"    Book source : {info.get('source')!r}  (UAS tagged 'Cplx*')\n"
            f"\n"
            f"    What is needed:\n"
            f"      - Identify which stmt(s) supply the inputs to compose this\n"
            f"        field (e.g. sum of Acct.X + Acct.Y, or BS.line minus IS.line).\n"
            f"      - Implement the composition either in the relevant stmtOBJ_Tax\n"
            f"        class or as a follow-up resolver in irs.Form1065 services.\n"
            f"      - Replace the bookNS 'Cplx*' UAS with the real path so this\n"
            f"        field migrates from complexDict to fillDict on the next run.\n"
            f"    '''\n"
            f"    raise NotImplementedError(\"{nm}: complex mapping deferred - see docstring.\")\n"
        )
    return '\n'.join(src_lines)

_stub_src = _emit_complex_stubs(complexDict)
if _stub_src:
    exec(_stub_src, globals())
    _stub_names = [n for n in globals() if n.startswith('_Cplx_')]
    print(f'Generated {len(_stub_names)} _Cplx_<fid>() stubs.')
    for n in _stub_names[:3]:
        print(f'\n=== {n} ===')
        print(globals()[n].__doc__)
else:
    print('No complex fields in this run -- no stubs emitted.')
    print('(Tag a bookNS_*.json UAS path with "Cplx.<topic>" to mark a field complex.)')


No complex fields in this run -- no stubs emitted.
(Tag a bookNS_*.json UAS path with "Cplx.<topic>" to mark a field complex.)


## 6. Done.

* `Form1065_FILL.pdf` is written under
  `pages/AccountingData/<yr>/YE_Tax_Records/Forms_IRS/`.
* `result['fillDict']` lists fields filled with real book values.
* `result['complexDict']` lists fields explicitly tagged complex —
  each gets a `_Cplx_<fid>()` stub awaiting follow-up work.
* `result['df']` is the full reconciliation table — every PDF field
  with its normalized `fid`, book source (if any), resolved value, and
  status (`filled` | `complex` | `blank`).

To improve coverage on the next iteration:

1. Open `Form1065_namespace.pdf` and `Form1065_FILL.pdf` side-by-side.
2. For each field that is wrong or missing, locate its `f###` /
   normalized `F###` fid.
3. Fix the relevant `bookNS_*.json` mapping (or the underlying
   `stmtOBJ` source-data) and re-run this notebook.

# Form 8825 Design

- the foundation for 8825 is based on a complex bookNS wher the namespace is dynamically created based on
    - stmtProfile - info on LLC
    - llcProperty - base info per property (name, address, status, etc...)
    - stmtIS - Income/Expense per property,
- Requirements
    - within llcExpRev transaction every income/expense transaction MUST be matched to a Property
    - Rental Income is proprated between Renters and Member's usage - Rev.Rent is classified by Customer/Member - FIXME
    - stmtIS_Tax.loadFillDict('Form8825') builds a dynamic bookNS that aggregates all Income/Expenses per Property
        - 8825 namespace form: <IS><PropNm><UAS>



Niebuhr's contributions to political philosophy include using the resources of theology to argue for political realism. His work has also significantly influenced international relations theory, leading many scholars to move away from idealism and embrace realism. A large number of scholars, including political scientists, political historians, and theologians, have noted his influence on their thinking. Aside from academics, activists such as Myles Horton and Martin Luther King Jr., and numerous politicians have also cited his influence on their thought,including Hillary Clinton, Hubert Humphrey, and Dean Acheson, as well as presidents Barack Obama and Jimmy Carter. Niebuhr has also influenced the Christian right in the United States. The Institute on Religion and Democracy, a conservative think tank founded in 1981, has adopted Niebuhr's concept of Christian realism on their social and political approaches